# Day 13: RNN — Reading One Word at a Time

**Goal:** Build the first sequence model — an RNN — and use it to generate names character by character.

### Where we are

```
Day 1-12: Networks that ignore word order
   "the cat sat"  ≈  "sat the cat"   (BoW & mean embedding both miss this)

Day 13: A sequence model that respects order
   Process tokens one at a time, carrying memory forward
```

### Plan for today

1. Build an RNN cell from scratch (the math, in code)
2. Use PyTorch's `nn.RNN` (the easy way)
3. Apply it to character-level **name generation**
4. Sample new names from the trained model
5. See the vanishing gradient problem with our own eyes
6. Compare to LSTM (next-gen RNN with memory gates)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

## 1. An RNN Cell From Scratch

The math of an RNN step:

```
h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b)
```

That's it. Let's write it as code:

In [ ]:
class SimpleRNNCell(nn.Module):
    """A single RNN step from scratch. h_t = tanh(W_xh @ x + W_hh @ h_prev + b)."""
    
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        # Combine x and h_prev into one linear layer for efficiency:
        # tanh(W_xh @ x + W_hh @ h + b) = tanh([W_xh, W_hh] @ [x; h] + b)
        self.linear = nn.Linear(input_size + hidden_size, hidden_size)
    
    def forward(self, x_t, h_prev):
        # x_t: (batch, input_size)
        # h_prev: (batch, hidden_size)
        combined = torch.cat([x_t, h_prev], dim=1)   # (batch, input+hidden)
        h_t = torch.tanh(self.linear(combined))       # (batch, hidden)
        return h_t

# Test it on a fake sequence
cell = SimpleRNNCell(input_size=5, hidden_size=8)

# Imagine a sentence of 4 tokens, each is a 5-dim embedding
batch_size = 2
seq_len = 4
inputs = torch.randn(batch_size, seq_len, 5)

# Process step by step
h = torch.zeros(batch_size, 8)        # start with empty memory
print(f"Initial hidden state: {h.shape}")
print(f"\nProcessing {seq_len} time steps...\n")

for t in range(seq_len):
    x_t = inputs[:, t, :]              # (batch, input_size) at step t
    h = cell(x_t, h)
    print(f"  Step {t}: input={x_t.shape}, new hidden={h.shape}")

print(f"\nFinal hidden state (summary of whole sequence):")
print(h)

### What just happened

We processed a sequence ONE TOKEN AT A TIME. The same `cell.linear` was used at every step, and the hidden state was passed along.

That's the entire trick. **One small operation, repeated for every position.**

---

## 2. The Same Thing with `nn.RNN`

PyTorch provides this pre-built. The math is identical, but it handles the loop and batching for you:

In [ ]:
rnn = nn.RNN(input_size=5, hidden_size=8, batch_first=True)
# batch_first=True means input shape is (batch, seq_len, features) — easier to think about

# Same fake data as before
inputs = torch.randn(2, 4, 5)         # (batch=2, seq_len=4, features=5)

# Just one call — PyTorch handles the loop internally
output, final_hidden = rnn(inputs)

print(f"Input shape:         {inputs.shape}     (batch, seq_len, features)")
print(f"Output shape:        {output.shape}     (batch, seq_len, hidden) — h at EVERY step")
print(f"Final hidden shape:  {final_hidden.shape}     (num_layers, batch, hidden) — JUST the last h")

print(f"\nNote: output[:, -1, :] (last step's h) should equal final_hidden:")
print(f"  Match: {torch.allclose(output[:, -1, :], final_hidden[0])}")

## 3. The Project — Character-Level Name Generator

We'll train an RNN to generate plausible names, one character at a time.

### The data: 30 fake names

In [ ]:
# A small set of names to train on

names = [
    "amit", "rohit", "sara", "priya", "neha", "ravi", "ankit", "deepa",
    "vikram", "anita", "raj", "kiran", "anjali", "rahul", "pooja",
    "manish", "rekha", "sunil", "asha", "varun", "lakshmi", "naveen",
    "shreya", "abhishek", "kavita", "arjun", "meera", "rakesh", "divya",
    "sanjay", "geeta", "vinod", "preeti", "ashok", "bhavna",
]

print(f"Total names: {len(names)}")
print(f"Sample: {names[:10]}")

# Build character vocab
chars = sorted(set(''.join(names)))
# Add special tokens: '.' = start/end marker
chars = ['.'] + chars
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
vocab_size = len(chars)

print(f"\nVocabulary ({vocab_size} chars): {chars}")
print(f"  '.' is our start/end marker")

### Prepare training examples

Each name becomes pairs of (input, target) characters. For "amit":

```
input  target
  .  →    a    (start, then "a")
  a  →    m
  m  →    i
  i  →    t
  t  →    .    (end of name)
```

We wrap with `.` to mark start/end.

In [ ]:
# Convert names to training data
# Each name: wrap with '.', then make input/target pairs

def name_to_indices(name):
    """'.amit.' → list of indices"""
    return [char_to_idx[c] for c in '.' + name + '.']

# Pad all to same length for batching
max_len = max(len(name) for name in names) + 2   # +2 for start and end markers
print(f"Max sequence length: {max_len}")

# Build training data
X_all = []
Y_all = []
for name in names:
    indices = name_to_indices(name)
    # Pad with '.' to fixed length
    while len(indices) < max_len:
        indices.append(char_to_idx['.'])
    # x = all but last char, y = all but first char (shifted by 1)
    X_all.append(indices[:-1])
    Y_all.append(indices[1:])

X = torch.tensor(X_all, dtype=torch.long)
Y = torch.tensor(Y_all, dtype=torch.long)

print(f"X shape: {X.shape}  (num_names × seq_len)")
print(f"Y shape: {Y.shape}")

# Show the first example
print(f"\nName: '{names[0]}'")
print(f"Padded indices: {X_all[0]}")
print(f"X (input):      {[idx_to_char[i] for i in X_all[0]]}")
print(f"Y (target):     {[idx_to_char[i] for i in Y_all[0]]}")
print(f"\nAt each step, X is the current char; Y is what should come next.")

### Define the model

```
input char ID  →  Embedding  →  RNN  →  Linear  →  next char probabilities
```

In [ ]:
class CharRNN(nn.Module):
    """Character-level RNN: input char → embedding → RNN → vocab logits."""
    
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        # At each time step, output a distribution over the vocabulary
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, h=None):
        # x: (batch, seq_len) — token IDs
        # h: optional initial hidden state — None means zeros
        embedded = self.embedding(x)             # (batch, seq_len, embed_dim)
        output, h = self.rnn(embedded, h)        # output: (batch, seq_len, hidden)
        logits = self.fc(output)                  # (batch, seq_len, vocab_size)
        return logits, h

# Build the model
torch.manual_seed(42)
model = CharRNN(vocab_size, embed_dim=16, hidden_dim=64)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

### Train the model

At each position, predict the next character. Loss = Cross-Entropy averaged across all positions.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

losses = []
for epoch in range(500):
    model.train()
    logits, _ = model(X)                       # logits: (batch, seq_len, vocab_size)
    
    # CrossEntropy wants (N, C) logits and (N,) targets — flatten over batch+seq
    loss = loss_fn(logits.reshape(-1, vocab_size), Y.reshape(-1))
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    
    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d}: loss={loss.item():.4f}")

print(f"\nFinal loss: {losses[-1]:.4f}")

plt.figure(figsize=(10, 4))
plt.plot(losses, 'b-', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.title('RNN training on character-level name data')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Generate New Names

Sampling new names step by step:
1. Start with `.` (the start token)
2. Feed it through the model → get probabilities for next character
3. Sample a character from those probabilities
4. Feed that character back in
5. Repeat until we sample `.` (end token) or hit max length

In [ ]:
def generate_name(model, max_len=20, temperature=1.0):
    """Generate a new name one character at a time."""
    model.eval()
    
    chars_generated = []
    # Start with '.' as the first input
    x = torch.tensor([[char_to_idx['.']]], dtype=torch.long)  # (1, 1)
    h = None
    
    with torch.no_grad():
        for _ in range(max_len):
            logits, h = model(x, h)            # logits: (1, 1, vocab_size)
            # Take the prediction from the LAST step
            last_logits = logits[0, -1, :] / temperature   # (vocab_size,)
            probs = F.softmax(last_logits, dim=0)
            # Sample from the distribution
            next_idx = torch.multinomial(probs, num_samples=1).item()
            next_char = idx_to_char[next_idx]
            
            if next_char == '.':              # end of name
                break
            chars_generated.append(next_char)
            
            # Feed this char back as the next input
            x = torch.tensor([[next_idx]], dtype=torch.long)
    
    return ''.join(chars_generated)

# Generate 15 new names
print("Generated names from the trained RNN:\n")
torch.manual_seed(0)
for i in range(15):
    name = generate_name(model)
    print(f"  {i+1:>2}. {name}")

### Temperature — controlling creativity

When sampling, we divided logits by `temperature`. This controls how "creative" vs "conservative" the generation is:

- `temperature = 1.0` → sample from the original distribution
- `temperature < 1.0` → sharpen: pick the most likely character more often (predictable)
- `temperature > 1.0` → flatten: random characters more often (creative/weird)

In [ ]:
# Effect of temperature on generation

for temp in [0.5, 1.0, 1.5]:
    print(f"\n--- Temperature = {temp} ---")
    torch.manual_seed(0)
    for _ in range(8):
        name = generate_name(model, temperature=temp)
        print(f"  {name}")

print("\nLow temperature  → safer, more like training data")
print("High temperature → more unusual, sometimes nonsense")
print("\nReal LLMs expose this same parameter — you've seen it in ChatGPT settings!")

## 5. Visualizing the Hidden State

The hidden state is the "memory." Let's see how it evolves as the RNN reads a name:

In [ ]:
# Feed a name through and watch the hidden state change
name = "amit"
chars = list('.' + name + '.')
indices = torch.tensor([[char_to_idx[c] for c in chars]], dtype=torch.long)

model.eval()
with torch.no_grad():
    embedded = model.embedding(indices)
    output, _ = model.rnn(embedded)            # (1, seq_len, hidden)

hidden_states = output[0].numpy()              # (seq_len, hidden)

# Plot as heatmap
plt.figure(figsize=(10, 5))
plt.imshow(hidden_states.T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(label='Hidden state value')
plt.xticks(range(len(chars)), chars, fontsize=12)
plt.xlabel('Input character at each time step')
plt.ylabel('Hidden state dimension (64 dims)')
plt.title(f"How the RNN's 'memory' evolves while reading '{name}'")
plt.show()

print("Each column = the hidden state AFTER reading that character.")
print("Notice how it changes step by step — that's the RNN building up understanding.")

## 6. The Vanishing Gradient — See It For Yourself

When we backprop through a long sequence, gradients can vanish. Let's prove it:

In [ ]:
# Feed sequences of varying length, measure gradient magnitude at the FIRST step
# If gradients vanish, early-step gradients should be tiny.

torch.manual_seed(0)
test_rnn = nn.RNN(input_size=10, hidden_size=20, batch_first=True)
test_proj = nn.Linear(20, 1)

seq_lengths = [5, 20, 50, 100, 200]
first_step_grads = []

for seq_len in seq_lengths:
    # Fresh input that we'll inspect the gradient of
    x = torch.randn(1, seq_len, 10, requires_grad=True)
    
    # Forward
    output, _ = test_rnn(x)
    loss = test_proj(output[:, -1, :]).sum()    # use only the LAST output
    
    # Backward
    loss.backward()
    
    # How big is the gradient on the FIRST step input?
    first_grad = x.grad[:, 0, :].abs().mean().item()
    last_grad = x.grad[:, -1, :].abs().mean().item()
    
    first_step_grads.append(first_grad)
    print(f"Seq len {seq_len:>3}: gradient at first step = {first_grad:.6f}, "
          f"at last step = {last_grad:.6f}")

print(f"\nNotice: as seq_len grows, the gradient at the FIRST step shrinks dramatically.")
print(f"This is why basic RNNs forget long-range dependencies.")

# Plot
plt.figure(figsize=(9, 4))
plt.plot(seq_lengths, first_step_grads, 'bo-', markersize=8, linewidth=2)
plt.xlabel('Sequence length')
plt.ylabel('Avg |gradient| at first step')
plt.title('Vanishing gradient: signal dies as sequence grows')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

## 7. LSTM — The Improvement

**LSTM** (Long Short-Term Memory) replaces the simple RNN update with "gates" that decide what to keep and what to forget. It dramatically reduces the vanishing gradient problem.

You don't need to derive the gates by hand — just use `nn.LSTM`. Same API as `nn.RNN`.

In [ ]:
class CharLSTM(nn.Module):
    """Same as CharRNN but with LSTM instead of RNN."""
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, h=None):
        embedded = self.embedding(x)
        # LSTM returns (output, (h, c)) — there's an extra "cell state"
        output, hc = self.lstm(embedded, h)
        return self.fc(output), hc

# Train RNN vs LSTM and compare
def train_model(model, X, Y, epochs=500, lr=0.01):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    losses = []
    for _ in range(epochs):
        out = model(X)
        logits = out[0]
        loss = loss_fn(logits.reshape(-1, vocab_size), Y.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

torch.manual_seed(42)
rnn_model = CharRNN(vocab_size, embed_dim=16, hidden_dim=64)
rnn_losses = train_model(rnn_model, X, Y)

torch.manual_seed(42)
lstm_model = CharLSTM(vocab_size, embed_dim=16, hidden_dim=64)
lstm_losses = train_model(lstm_model, X, Y)

plt.figure(figsize=(10, 4))
plt.plot(rnn_losses, 'b-', label=f'RNN (final {rnn_losses[-1]:.3f})', linewidth=2)
plt.plot(lstm_losses, 'r-', label=f'LSTM (final {lstm_losses[-1]:.3f})', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('RNN vs LSTM — same task, different architecture')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"RNN params:  {sum(p.numel() for p in rnn_model.parameters()):,}")
print(f"LSTM params: {sum(p.numel() for p in lstm_model.parameters()):,}  (more — has gates)")
print(f"\nLSTM usually gets lower loss and is more robust for long sequences.")

---

## Exercises

1. **More names:** Add 20 more names to the training data. Does generation improve?

2. **Bigger hidden:** Set `hidden_dim=128`. Does the loss go lower? Are generated names better?

3. **LSTM vs RNN sampling:** Use the trained LSTM to generate names. Are they more name-like than the basic RNN?

4. **Train on Shakespeare:** Replace `names` with first few lines of Shakespeare. Train the same model. What does it generate? (Hint: it'll look like fake Shakespeare characters)

5. **Vanishing gradient experiment:** Run the vanishing-gradient cell with an LSTM instead of a basic RNN. Does the gradient still vanish as fast?

---

## Key Takeaways

### What an RNN does

```
For each step t:
  h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b)

The hidden state h carries forward through the sequence,
accumulating "memory" of everything read so far.
```

### Critical differences from earlier days

| | Before (Day 10-12) | Today (Day 13) |
|---|---|---|
| Word order | Ignored | Respected |
| Operation | One pass | Loop over sequence |
| Memory | None | Hidden state |
| Input length | Fixed (padded) | Naturally variable |

### When to use what

| Task | Best architecture |
|------|-------------------|
| Time series, simple sequences | RNN/LSTM/GRU |
| Long-range dependencies | Transformer (or LSTM) |
| Modern NLP | Transformer |
| Edge devices, real-time | RNN/GRU (faster) |

### The bigger picture

```
Days 1-12:  Feed-forward + mean embeddings → no order
Day 13:     RNN → respects order, but vanishing gradients
Day 14-18:  Bigram LM → simple language modeling task
Day 15+:    Attention → fixes RNN's long-range problem
Day 19+:    Transformer → attention done right
```

You've crossed a major threshold today. From here on, every model we touch is a **sequence model**.

**Tomorrow:** Day 14 — Bigram Language Model. The simplest "predict the next word" model. It's GPT's great-grandparent.